<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 10 · 用 Middleware 给真实 Agent 接上记忆

应用保存了一条刚确认的项目验收代号。接下来启动一个没有旧聊天记录的 Agent，让 middleware 自动准备上下文。我们既看最终回答，也用回调检查真正发给模型的消息，再用没有历史的模型和另一个项目做对照。最后用 Scope binding 解析项目，并打开一次 `auto_capture`。

**完成后你能做到：** 通过现有 LangChain Middleware 接入真实模型，区分写入、召回注入、回答和会话采集；用 binding 选择 Scope；确认采集到的是 Source 而不是 Memory。

预计 25 分钟。先按 [README](README.md) 安装环境；本篇可以独立运行，不依赖其他 Notebook 的变量或数据。需要真实模型配置。

按顺序读说明、运行代码，再对照结果。练习可以改输入；完整重跑使用 **Restart Kernel & Run All**。

## 准备本篇实验

这格启动一个回环地址的真实 Server，并创建独立 Scope，默认把数据保存在本篇自己的 SQLite 文件中，也可按 [README](README.md#使用-oceanbase-运行) 显式选择专用 OceanBase 测试库。
`_tutorial.py` 只管理环境和显示结果；下面的业务调用都是可在应用中复用的公开 API。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))


if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("10", features=())
client = lab.client
assert client is not None

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 10",
        summary="第 10 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-10",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 应用先保存一条刚确认的事实

随机代号在本次运行中才生成，模型不可能从训练数据知道它。这里由应用使用日常 Memory API 显式写入，
便于把“写入成功”和后面的“模型收到历史”分别检查。

Middleware 默认只做召回。后面会单独打开 `auto_capture=True`，把成功回合采集为 Source；
采集成功不等于已经写入 Memory，提取仍是第 08 篇的能力。

In [ ]:
from uuid import uuid4

from powercontext.http import ListMemoryEntriesRequest, RememberMemoryRequest

project_code = "CSV-" + uuid4().hex[:10].upper()
fact = f"release_codename: 本项目订单 CSV 导入器的验收代号是 {project_code}。"
saved = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="decision",
        text=fact,
        reason="用户确认的本次项目代号",
    )
)
entries = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert saved.entry is not None and any(project_code in entry.text for entry in entries.entries)
show({"已保存的项目事实": fact})

## 2. 几行接入现有 Agent

`create_agent` 创建 LangChain Agent，`PowerContextMiddleware()` 在每次模型调用前准备上下文。
PowerContextScope 提供本次调用的服务和 Scope，应用负责选择正确项目。
本篇不写自定义召回节点，也不手工拼装 PowerContext 上下文。

In [ ]:
from _tutorial import chat_settings
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from powercontext_langchain import PowerContextMiddleware, PowerContextScope

model = ChatOpenAI(**chat_settings())
system_prompt = "协助开发订单导入器。回答项目约定时给出已有依据；代号需完整复述，不知道时明确说明。"
agent = create_agent(
    model,
    tools=[],
    system_prompt=system_prompt,
    middleware=[PowerContextMiddleware()],
    context_schema=PowerContextScope,
)
agent_scope = PowerContextScope(scope_id=scope_id, base_url=lab.base_url)

## 3. 先问一个没有记忆的模型

使用同一个真实模型、同一个系统提示和问题，但不连接 PowerContext，也不传入上面的事实。
记录这个对照结果，不用它猜得像不像来替代后面对模型输入的检查。

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

question = "release_codename: 本项目订单 CSV 导入器的验收代号是什么？请完整给出，没有依据时说明。"
baseline = await model.ainvoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
baseline_contains_code = project_code in str(baseline.content)
print(baseline.content)

## 4. 检查真正送给模型的消息

下面的回调只观察模型调用，不替换模型输出。Agent 输入只有当前问题，没有之前的会话。
运行后查看真实输入中是否出现 PowerContext 历史块，以及本次随机生成的代号。

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class InspectModelInput(BaseCallbackHandler):
    def __init__(self):
        self.calls = []

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls.extend([list(batch) for batch in messages])


inspector = InspectModelInput()
second = await agent.ainvoke(
    {"messages": [("user", question)]},
    context=agent_scope,
    config={"callbacks": [inspector]},
)
assert inspector.calls, "没有观察到真实模型调用。"
table([{"角色": message.type, "真实输入预览": str(message.content)[:900]} for message in inspector.calls[0]])

## 5. 分别检查注入、回答和会话状态

输入中确实有代号，才能证明召回材料送达；最终答案出现代号，才说明这次模型使用了它。
同时查看返回的会话消息：middleware 添加的历史块不应被写回聊天状态。

In [ ]:
input_texts = [str(message.content) for message in inspector.calls[0]]
supplied = any("PowerContext host-supplied context" in text and project_code in text for text in input_texts)
answers = [message for message in second["messages"] if message.type == "ai" and not getattr(message, "tool_calls", [])]
assert answers
answer = str(answers[-1].content)
history_clean = all("PowerContext host-supplied context" not in str(message.content) for message in second["messages"])
table([
    {"观察点": "没有历史的对照回答", "包含随机代号": baseline_contains_code},
    {"观察点": "第二个会话的真实模型输入", "包含随机代号": supplied},
    {"观察点": "第二个会话最终回答", "包含随机代号": project_code in answer},
])
print(answer)
assert supplied and project_code in answer and history_clean
print("召回材料没有积累进返回的会话历史。")

## 练习：同一个 Agent，换一个没有该记忆的项目

创建新 Scope，再用同一个 Agent 对象和相同问题调用。它仍只收到当前问题。
同时检查实际模型输入与最终回答都没有原项目代号，避免仅凭“回答没提到”就断定没有注入。

In [ ]:
from powercontext.http import PrepareContextRequest

other = await client.create_scope(
    CreateScopeRequest(
        title="另一份订单导入器",
        summary="没有旧项目代号",
        idempotency_key=f"{lab.run_id}:other-agent-project",
    )
)
empty = await client.prepare_context(
    PrepareContextRequest(scope_id=other.scope_id, query="release_codename", max_bytes=4000)
)
assert empty.status == "empty"
other_inspector = InspectModelInput()
other_answer = await agent.ainvoke(
    {"messages": [("user", question)]},
    context=PowerContextScope(scope_id=other.scope_id, base_url=lab.base_url),
    config={"callbacks": [other_inspector]},
)
assert other_inspector.calls
assert all(project_code not in str(message.content) for batch in other_inspector.calls for message in batch)
other_answers = [
    message for message in other_answer["messages"] if message.type == "ai" and not getattr(message, "tool_calls", [])
]
assert other_answers
final_text = str(other_answers[-1].content)
assert project_code not in final_text
print(final_text)

## 6. 用 binding 记住“这个工作区对应哪个项目”

Agent 集成通常不让模型挑选 `scope_id`。应用把工作区、会话等外部身份绑到 Scope，之后用 binding 解析，
再把解析结果交给 Middleware。显式传入的 `scope_id` 仍然优先。

In [ ]:
from powercontext.http import ResolveScopeBindingRequest, SetScopeBindingRequest

workspace_key = {"integration": "tutorial", "kind": "workspace", "external_id": str(lab.directory)}
binding = await client.set_scope_binding(
    SetScopeBindingRequest.model_validate({"key": workspace_key, "scope_id": scope_id})
)
resolved = await client.resolve_scope_binding(
    ResolveScopeBindingRequest.model_validate({"binding_keys": [workspace_key]})
)
explicit = await client.resolve_scope_binding(
    ResolveScopeBindingRequest.model_validate({"explicit_scope_id": other.scope_id, "binding_keys": [workspace_key]})
)
assert binding.scope_id == resolved.scope_id == scope_id
assert explicit.scope_id == other.scope_id
show({
    "工作区绑定到": resolved.scope_id,
    "显式 Scope 优先": explicit.scope_id,
})

## 7. 成功回合可以采集为 Source，但不会自动变成 Memory

再创建一个打开 `auto_capture=True` 的 Agent。运行成功后，最新用户消息和最终回答会写成 Content Source。
下面按 Middleware 使用的确定性 Source ID 读回这条材料，并确认日常 Memory 条目没有增加。

Host 插件、MCP 工具和 LangGraph Memory 工具不在本课展开；需要时见
[接口说明](../../docs/zh/docs/develop/interfaces.md)、[LangGraph 集成](../../integrations/langgraph/README.md)
和 [LangChain auto_capture](../../integrations/langchain/README.md)。

In [ ]:
import hashlib

capturing_agent = create_agent(
    model,
    tools=[],
    system_prompt=system_prompt,
    middleware=[PowerContextMiddleware(auto_capture=True)],
    context_schema=PowerContextScope,
)
capture_question = "请完整复述本项目的验收代号，并说明这只是历史材料。"
captured_run = await capturing_agent.ainvoke(
    {"messages": [("user", capture_question)]},
    context=PowerContextScope(scope_id=scope_id, base_url=lab.base_url),
)
captured_answers = [
    message for message in captured_run["messages"] if message.type == "ai" and not getattr(message, "tool_calls", [])
]
assert captured_answers
assistant_text = captured_answers[-1].text.strip()
digest = hashlib.sha256()
for value in (scope_id, capture_question, assistant_text):
    digest.update(value.encode("utf-8"))
    digest.update(b"\x00")
source_id = f"langchain-agent-turn-{digest.hexdigest()}"
captured_source = await client.get_source(scope_id, "content", source_id)
assert capture_question in str(captured_source.content)
after_capture = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert [entry.citation for entry in after_capture.entries] == [entry.citation for entry in entries.entries]
show({
    "采集到的 Source": captured_source.source_id,
    "其中包含用户问题": True,
    "Memory 条目未增加": True,
})
print(str(captured_source.content)[:800])

## 保存收获，关闭连接

应用显式保存事实，Middleware 按当前 Scope 注入本次请求的上下文。binding 负责选对项目，`auto_capture` 只保存回合 Source。
MCP、Host 插件和 LangGraph 工具型 Memory 有单独文档，本课不演示。

下面关闭本篇 Client 和 Server，保留实验文件供检查。中途停止时也可运行这一格；清理方式见 [README](README.md#清理实验数据)。

下一篇：[11](11_http_api_lifecycle.ipynb)。

## 补充实验：让 Agent 自己选择写入工具

Middleware 已经展示自动注入与 auto_capture。现在我们给同一个真实对话模型提供正式 Memory 工具，并明确授权它保存一条新规则。应看到模型发出 tool_call，再由另一次公开读取确认结果；auto_capture 本身不会完成这次 Memory 写入。

本实验需要模型支持工具调用，完整 MCP 会话与工具接入见第 19 篇。

In [ ]:
from powercontext_langgraph import PowerContextScope as ToolScope
from powercontext_langgraph import powercontext_tools

from powercontext.http import SearchMemoryRequest

tool_agent = create_agent(
    model,
    tools=powercontext_tools(),
    context_schema=ToolScope,
    system_prompt="Save only user-confirmed project rules through the actual tool. Report the tool result accurately.",
)
written = await tool_agent.ainvoke(
    {
        "messages": [
            ("user", "请记住我已确认的长期规则：export_encoding: 本项目 CSV 导出使用 UTF-8。请实际调用记忆工具保存。")
        ]
    },
    context=ToolScope(scope_id=scope_id, base_url=lab.base_url),
)
actual_calls = [call for message in written["messages"] for call in getattr(message, "tool_calls", [])]
assert any(call["name"] == "powercontext_remember" for call in actual_calls)
stored_rule = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="export_encoding"))
assert any("UTF-8" in hit.text for hit in stored_rule.hits)
table([{"工具": call["name"], "实际参数": call["args"]} for call in actual_calls])

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")